# SAXS Tensor Tomography — Method Comparison

Compares four reconstruction methods on the same frogbone dataset, all converted to the
same representation: one scalar volume per q-direction `(K, X, Y, Z)`.

| Section | Method | Library |
|---------|--------|---------|
| 1 | FBP | smartt |
| 2 | NearestNeighbor + LBFGS | mumott |
| 3 | SphericalHarmonics + LBFGS → projected to NN | mumott |
| 4 | GaussianKernels + LBFGS → projected to NN | mumott |
| 5 | Side-by-side comparison + missing-cone visualisation | — |
| 6 | IsoNet missing-wedge correction | smartt |

All mumott reconstructions use the same optimiser settings so comparisons are fair.

Before the comparison section all reconstructions are **spherically cropped** to the
largest inscribed sphere (diameter = largest multiple of 8 ≤ min(X,Y,Z)) so every method
is evaluated on the same spatial region.

In [ ]:
try:
    from mumott.data_handling import DataContainer
except:
    !sh ../scripts/setup.sh
    !pip install --upgrade coverage pytest-cov

In [ ]:
import lovely_tensors as lt
lt.monkey_patch()

In [ ]:
import sys
sys.path.insert(0, '/myhome/smartt')

import pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt

from mumott.data_handling import DataContainer
from mumott.methods.basis_sets import SphericalHarmonics, GaussianKernels, NearestNeighbor
from mumott.methods.projectors import SAXSProjectorCUDA, SAXSProjector
from mumott.methods.residual_calculators import GradientResidualCalculator
from mumott.optimization.loss_functions import SquaredLoss
from mumott.optimization.optimizers import LBFGS
from mumott.optimization.regularizers import Laplacian
from mumott.core.probed_coordinates import ProbedCoordinates

from smartt.saxs_fbp import fibonacci_hemisphere, saxs_fbp_reconstruction, saxs_gd_reconstruction

# ── Parameters ────────────────────────────────────────────────────────────────
# DATA_PATH        = '/myhome/data/smartt/shared/frogbone/dataset_qbin_0009.h5'
DATA_PATH = '/myhome/data/smartt/shared/b411/dataset_b411R_inf_1_0.220_1.900.h5'
K                = 30      # number of q-directions (Fibonacci hemisphere)
HALF_SPACE       = 'y'     # y>0 hemisphere to match goniometer frame
LBFGS_MAXITER    = 20
LAPLACIAN_WEIGHT = 1e-1
SH_ELL_MAX       = 8

# ── Cache settings ─────────────────────────────────────────────────────────────
# mumott optimisations are saved as .npy files (keyed by method + K) so they
# can be skipped on subsequent runs.  Set force_reload=True to rerun from scratch.
CACHE_DIR    = pathlib.Path('/myhome/data/smartt/shared/results/saxs_comparison')
force_reload = False

CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Cache directory: {CACHE_DIR}')
print(f'force_reload: {force_reload}')

In [ ]:
dc = DataContainer(DATA_PATH, nonfinite_replacement_value=0)
dc.geometry.full_circle_covered = False

geometry = dc.geometry

print('data shape    :', dc.data.shape, '  (N, J, K_det, M)')
print('volume_shape  :', geometry.volume_shape)
print('n_projections :', len(dc.projections))

y_directions = fibonacci_hemisphere(K, half_space=HALF_SPACE)
print(f'\ny_directions shape: {y_directions.shape}  (K, 3)')

## 1. smartt FBP Reconstruction

For each of the K q-directions the scalar sinogram is built via a GPU einsum over the
arc-fraction projection matrix, then inverted with ramp-filtered backprojection (ASTRA BP3D).

In [ ]:
ball_threshold = 0.2

recon_fbp, y_dirs, nn_helper = saxs_fbp_reconstruction(
    dc=dc,
    k_fibonacci=K,
    filter_type='hann',
    n_projection_samples=64,
    device=device,
    verbose=True,
    return_matrix=True,
    projection_method='ball',
    ball_threshold=ball_threshold,
    half_space=HALF_SPACE,
)


# recon_fbp, y_dirs, nn_helper = saxs_gd_reconstruction(
#     dc=dc,
#     k_fibonacci=K,
#     n_iterations=300,
#     lr=0.5,
#     n_projection_samples=64,
#     device=device,
#     verbose=True,
#     return_matrix=True,
#     projection_method="ball",
#     ball_threshold=ball_threshold,
#     half_space=HALF_SPACE,
# )
recon_fbp_np = recon_fbp.cpu().numpy()   # (K, X, Y, Z)
print(f'\nFBP shape  : {recon_fbp_np.shape}')
print(f'value range: [{recon_fbp_np.min():.3e}, {recon_fbp_np.max():.3e}]')

## 2. mumott NearestNeighbor Reconstruction

Uses the same K Fibonacci directions as the FBP.  Each coefficient `C[x,y,z,k]` directly
represents the scattering intensity at direction `y_k` — no basis conversion needed.

In [ ]:
# Build the shared mumott projector once (fast — geometry setup only).
# Always constructed so it is available if any method needs to optimise.
projector_mumott = (
    SAXSProjector(geometry) if torch.cuda.is_available() else SAXSProjector(geometry)
)

_nn_cache = CACHE_DIR / f'mumott_nn_K{K}.npy'

if not force_reload and _nn_cache.exists():
    print(f'Loading NN coefficients from cache: {_nn_cache}')
    coeffs_nn = np.load(_nn_cache)
else:
    basis_nn = NearestNeighbor(
        directions=y_directions,
        probed_coordinates=geometry.probed_coordinates,
    )
    rc_nn   = GradientResidualCalculator(data_container=dc, basis_set=basis_nn, projector=projector_mumott)
    loss_nn = SquaredLoss(rc_nn)
    loss_nn.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    opt_nn  = LBFGS(loss_nn, maxiter=LBFGS_MAXITER)
    result_nn = opt_nn.optimize()
    coeffs_nn = result_nn['x']                    # (X, Y, Z, K)
    np.save(_nn_cache, coeffs_nn)
    print(f'Saved NN coefficients → {_nn_cache}')

print(f'coeffs_nn shape: {coeffs_nn.shape}')

In [ ]:
recon_nn = np.moveaxis(coeffs_nn, -1, 0)   # (K, X, Y, Z)
print(f'NN reconstruction shape: {recon_nn.shape}')

## 3. mumott SphericalHarmonics Reconstruction → projected to NN

Optimises a band-limited SH expansion (`ell_max=6`, C=28 coefficients with Friedel symmetry),
then evaluates it at the K Fibonacci directions to obtain the same `(K, X, Y, Z)` layout.

The evaluation uses the SH basis matrix `B[k,c] = Y_c(y_k)` computed via
`_get_projection_matrix` — the same internal function used by mumott's `generate_map`.

In [ ]:
_sh_cache = CACHE_DIR / f'mumott_sh_ell{SH_ELL_MAX}_K{K}.npy'

# basis_sh is always instantiated: it is needed for the transfer step (cell below)
# regardless of whether the coefficients come from cache or from optimisation.
basis_sh = SphericalHarmonics(
    ell_max=SH_ELL_MAX,
    probed_coordinates=geometry.probed_coordinates,
)
print(f'SH basis: ell_max={SH_ELL_MAX}, n_coefficients={len(basis_sh)}')

if not force_reload and _sh_cache.exists():
    print(f'Loading SH coefficients from cache: {_sh_cache}')
    coeffs_sh = np.load(_sh_cache)
else:
    rc_sh   = GradientResidualCalculator(data_container=dc, basis_set=basis_sh, projector=projector_mumott)
    loss_sh = SquaredLoss(rc_sh)
    loss_sh.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    opt_sh  = LBFGS(loss_sh, maxiter=LBFGS_MAXITER)
    result_sh = opt_sh.optimize()
    coeffs_sh = result_sh['x']                    # (X, Y, Z, C)
    np.save(_sh_cache, coeffs_sh)
    print(f'Saved SH coefficients → {_sh_cache}')

print(f'coeffs_sh shape: {coeffs_sh.shape}')

In [ ]:
# Build evaluation matrix: B[k, c] = Y_c(y_k)  shape (K, C)
pc_eval = ProbedCoordinates()
pc_eval.vector = y_directions[:, np.newaxis, np.newaxis, :]   # (K, 1, 1, 3)
eval_sh = basis_sh._get_projection_matrix(pc_eval)[:, 0, 0, :]  # (K, C)

recon_sh  = np.einsum('xyzc, kc -> kxyz', coeffs_sh, eval_sh)  # (K, X, Y, Z)
print(f'SH→NN reconstruction shape: {recon_sh.shape}')

In [ ]:
plt.imshow(recon_sh[1][:, 20])

In [ ]:
plt.imshow(coeffs_sh[:, 20, :, 0])

In [ ]:
plt.imshow(recon_sh[1][:, :].sum(axis=1))

## 3.2. smartt NAF Reconstruction with SH → projected to NN

In [ ]:
from smartt.saxs_naf import saxs_naf_reconstruction, evaluate_models, plot_rsm_direction

_naf_cache = CACHE_DIR / f'mumott_naf_ell{SH_ELL_MAX}_K{K}.npy'

# basis_sh is always instantiated: it is needed for the transfer step (cell below)
# regardless of whether the coefficients come from cache or from optimisation.
basis_sh = SphericalHarmonics(
    ell_max=SH_ELL_MAX,
    probed_coordinates=geometry.probed_coordinates,
)
print(f'SH basis: ell_max={SH_ELL_MAX}, n_coefficients={len(basis_sh)}')

if not force_reload and _naf_cache.exists():
    print(f'Loading SH coefficients from cache: {_naf_cache}')
    coeffs_sh = np.load(_naf_cache)
else:
    res = saxs_naf_reconstruction(dc, ell_max=8, n_iterations=500, batch_size=80, reg_weight_tv=1.0)
    coeffs_naf = res["reconstruction"]   
    np.save(_naf_cache, coeffs_naf)
    print(f'Saved SH coefficients → {_naf_cache}')

res = saxs_naf_reconstruction(dc, ell_max=8, n_iterations=2000, batch_size=100, reg_weight_tv=1e-4, reg_weight_sh=500)
coeffs_naf = res["reconstruction"]   
np.save(_naf_cache, coeffs_naf)
print(f'Saved SH coefficients → {_naf_cache}')
print(f'coeffs_naf shape: {coeffs_naf.shape}')

# Build evaluation matrix: B[k, c] = Y_c(y_k)  shape (K, C)
pc_eval = ProbedCoordinates()
pc_eval.vector = y_directions[:, np.newaxis, np.newaxis, :]   # (K, 1, 1, 3)
eval_sh = basis_sh._get_projection_matrix(pc_eval)[:, 0, 0, :]  # (K, C)

recon_naf  = np.einsum('xyzc, kc -> kxyz', coeffs_naf, eval_sh)  # (K, X, Y, Z)
print(f'SH→NN reconstruction shape: {recon_naf.shape}')

  1%|▏         | 28/2000 [00:22<26:24,  1.24it/s, data=9.337e+08, loss=9.339e+08, lr=5.80e-03, sh=2.541e+05, tv=7.862e+00]

In [ ]:
with torch.no_grad():
    c = res["model"]()
    sh_raw = float(res["model"].sh_regularization(c))
    tv_raw = float(res["model"].tv_regularization(c))
    print(f"data loss:  7.675e+07")
    print(f"SH  (raw):  {sh_raw:.3e}  → weight for 1%: {7.675e5 / sh_raw:.2e}")
    print(f"TV  (raw):  {tv_raw:.3e}  → weight for 1%: {7.675e5 / tv_raw:.2e}")


## 4. mumott GaussianKernels Reconstruction → projected to NN

Same pipeline as SH but using isotropic Gaussian kernels on the sphere.  The default
`grid_scale=4` places kernels at a Kurihara mesh resolution of ~4 points per 90°.

In [ ]:
_gk_cache = CACHE_DIR / f'mumott_gk_K{K}.npy'

basis_gk = GaussianKernels(
    probed_coordinates=geometry.probed_coordinates,
)
print(f'GK basis: n_kernels={len(basis_gk)}')

if not force_reload and _gk_cache.exists():
    print(f'Loading GK coefficients from cache: {_gk_cache}')
    coeffs_gk = np.load(_gk_cache)
else:
    rc_gk   = GradientResidualCalculator(data_container=dc, basis_set=basis_gk, projector=projector_mumott)
    loss_gk = SquaredLoss(rc_gk)
    loss_gk.add_regularizer('laplacian', Laplacian(), regularization_weight=LAPLACIAN_WEIGHT)
    opt_gk  = LBFGS(loss_gk, maxiter=LBFGS_MAXITER)
    result_gk = opt_gk.optimize()
    coeffs_gk = result_gk['x']                    # (X, Y, Z, C_gk)
    np.save(_gk_cache, coeffs_gk)
    print(f'Saved GK coefficients → {_gk_cache}')

print(f'coeffs_gk shape: {coeffs_gk.shape}')

In [ ]:
eval_gk = basis_gk._get_projection_matrix(pc_eval)[:, 0, 0, :]  # (K, C_gk)
recon_gk  = np.einsum('xyzc, kc -> kxyz', coeffs_gk, eval_gk)  # (K, X, Y, Z)
print(f'GK→NN reconstruction shape: {recon_gk.shape}')

In [ ]:
# ## Spherical crop — apply to all methods before comparison
# # Crops each (K, X, Y, Z) reconstruction to (K, d, d, d) and zeros the sphere
# # exterior, so all methods are compared on the same spatial region.
# # d = (min(X, Y, Z) // 8) * 8  — for frogbone (65, 82, 65) this gives d = 64.
# from smartt.saxs_isonet.preprocess import spherical_crop, make_sphere_mask

# _t, _, SPHERE_D = spherical_crop(torch.from_numpy(recon_fbp_np), cube_size=None)
# recon_fbp_np = _t.numpy()

# recon_nn = spherical_crop(torch.from_numpy(recon_nn.astype(np.float32)), cube_size=SPHERE_D)[0].numpy()
# recon_sh = spherical_crop(torch.from_numpy(recon_sh.astype(np.float32)), cube_size=SPHERE_D)[0].numpy()
# recon_gk = spherical_crop(torch.from_numpy(recon_gk.astype(np.float32)), cube_size=SPHERE_D)[0].numpy()

# sphere_mask_np = make_sphere_mask(SPHERE_D)   # (d, d, d) bool — for masked percentiles
# print(f'Sphere diameter d={SPHERE_D}; all reconstructions now {recon_fbp_np.shape}')

## 5. Comparison

All four reconstructions are now in `(K, X, Y, Z)` float32 arrays with the same K Fibonacci
q-directions.  The following panels compare them using the same colour scale.

- **5.1** Mean intensity orthoslices
- **5.2** Single q-direction volume slices
- **5.3** Difference maps relative to FBP
- **5.4** Missing-cone visualisation in Fourier space

In [ ]:
reconstructions = {
    # 'FBP (smartt)':               recon_fbp_np,
    'NAF (smartt)':               recon_naf,
    # 'NearestNeighbor (mumott)':   recon_nn,
    # 'SphericalHarmonics (mumott)': recon_sh,
    'GaussianKernels (mumott)':   recon_gk,
}

vol_shape = recon_fbp_np.shape[1:]
mid_x, mid_y, mid_z = [s // 2 for s in vol_shape]
print('Volume shape:', vol_shape)
print(f'Central slices: x={mid_x}, y={mid_y}, z={mid_z}')

### 5.1 Mean intensity orthoslices

Colour scale is fixed to the [2nd, 98th] percentile of the FBP mean volume so all panels
are directly comparable.

In [ ]:
mean_fbp = recon_fbp_np.mean(axis=0)
# Use sphere interior for percentiles so exterior zeros don't compress the colour scale.
vmin, vmax = np.percentile(mean_fbp, [2, 98])

n_methods = len(reconstructions)
fig, axes = plt.subplots(n_methods, 3, figsize=(15, 4 * n_methods))

for row, (name, recon) in enumerate(reconstructions.items()):
    mean_vol = recon.mean(axis=0)
    slices   = [mean_vol[mid_x, :, :], mean_vol[:, mid_y, :], mean_vol[:, :, mid_z]]
    titles   = [f'YZ  x={mid_x}', f'XZ  y={mid_y}', f'XY  z={mid_z}']
    for col, (sl, title) in enumerate(zip(slices, titles)):
        im = axes[row, col].imshow(sl, cmap='inferno', vmin=vmin, vmax=vmax)
        axes[row, col].axis('off')
        axes[row, col].set_title(title, fontsize=10)
    axes[row, 0].set_ylabel(name, fontsize=9)

plt.suptitle('Mean scattering intensity — same colour scale (sphere interior percentiles)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Single q-direction volume slices

Each panel shows one reconstructed direction k.  Change `K_INSPECT` to explore other directions.

In [ ]:
K_INSPECT = -1  # change to inspect a different direction

r_fbp_k = recon_fbp_np[K_INSPECT]
vmin_k  = r_fbp_k.min()
vmin_k = 0
vmax_k  = np.percentile(r_fbp_k, 99)

fig, axes = plt.subplots(n_methods, 3, figsize=(15, 4 * n_methods))
for row, (name, recon) in enumerate(reconstructions.items()):
    vol    = recon[K_INSPECT]
    slices = [vol[mid_x, :, :], vol[:, mid_y, :], vol[:, :, mid_z]]
    titles = [f'{name} YZ  x={mid_x}', f'{name} XZ  y={mid_y}', f'{name}  XY  z={mid_z}']
    for col, (sl, title) in enumerate(zip(slices, titles)):
        im = axes[row, col].imshow(sl, cmap='inferno', vmin=vmin_k, vmax=vmax_k)
        axes[row, col].axis('off')
        axes[row, col].set_title(title, fontsize=10)
    axes[row, 0].set_ylabel(name, fontsize=10)
    

# plt.colorbar(im, ax=axes[:, 2], shrink=0.5, label='Intensity')
plt.suptitle(
    f'Direction k={K_INSPECT}  y={np.round(y_directions[K_INSPECT], 3)}',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

### 5.3 Difference maps (method − FBP)

Diverging colour map centred at zero; symmetric limits at the 98th percentile of |diff|.
Blue = method gives less intensity than FBP; red = more.

In [ ]:
others = {k: v for k, v in reconstructions.items() if k != 'FBP (smartt)'}
n_others = len(others)

fig, axes = plt.subplots(n_others, 3, figsize=(15, 4 * n_others))
if n_others == 1:
    axes = axes[np.newaxis]

for row, (name, recon) in enumerate(others.items()):
    diff   = recon.mean(axis=0) - mean_fbp
    absmax = np.percentile(np.abs(diff), 98)
    slices = [diff[mid_x, :, :], diff[:, mid_y, :], diff[:, :, mid_z]]
    titles = [f'YZ  x={mid_x}', f'XZ  y={mid_y}', f'XY  z={mid_z}']
    for col, (sl, title) in enumerate(zip(slices, titles)):
        im = axes[row, col].imshow(sl, cmap='RdBu_r', vmin=-absmax, vmax=absmax)
        axes[row, col].axis('off')
        axes[row, col].set_title(title, fontsize=10)
    axes[row, 0].set_ylabel(f'{name}\n− FBP', fontsize=9)
    plt.colorbar(im, ax=axes[row, 2], shrink=0.8, label='Δ intensity')

plt.suptitle('Difference in mean intensity: method − FBP (smartt)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5.4 Missing-cone visualisation in Fourier space

For a sub-CT at direction `y_k`, only beams orthogonal to `y_k` are used.  Fourier frequencies
along `y_k` have no contributing projections and remain unrecovered — the *missing cone*.
This appears as suppressed power (dark stripes) in the log-magnitude FFT of the sub-volume.

Iterative methods (NN, SH, GK) are expected to exhibit the same cone unless explicitly
constrained; FBP is particularly prone to streak artefacts from missing frequencies.

In [ ]:
# Log-magnitude FFT central slices for direction K_INSPECT
fig, axes = plt.subplots(2, n_methods, figsize=(5 * n_methods, 9))

for col, (name, recon) in enumerate(reconstructions.items()):
    vol_k   = recon[K_INSPECT].astype(np.float32)
    fft_vol = np.abs(np.fft.fftshift(np.fft.fftn(vol_k)))
    log_fft = np.log1p(fft_vol)
    fx, fy, fz = [s // 2 for s in fft_vol.shape]

    # Row 0: XZ plane (kx–kz), Row 1: YZ plane (ky–kz)
    for row, (sl, ylabel) in enumerate([
        (log_fft[:, fy, :], 'kx'),
        (log_fft[fx, :, :], 'ky'),
    ]):
        axes[row, col].imshow(sl, cmap='hot', aspect='auto',
                              vmin=np.percentile(sl, 5), vmax=np.percentile(sl, 99))
        axes[row, col].set_xlabel('kz', fontsize=8)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(ylabel, fontsize=8)
        axes[row, col].set_title(name, fontsize=8, wrap=True)

plt.suptitle(
    f'log|FFT| of direction k={K_INSPECT}  y={np.round(y_directions[K_INSPECT], 3)}\n'
    'Row 1: XZ  |  Row 2: YZ — missing cone = dark (suppressed) wedge',
    fontsize=10,
)
plt.tight_layout()
plt.show()

### 5.5 Per-direction mean intensity profile

For each direction k, the spatially-averaged intensity gives a scalar measure of how much
scattering power was reconstructed in that direction.  Comparing profiles across methods
reveals systematic biases (e.g. over-smoothing in SH at high-k directions).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
markers = ['o', 's', '^', 'D']
for (name, recon), marker in zip(reconstructions.items(), markers):
    profile = recon.mean(axis=(1, 2, 3))   # (K,) — mean over voxels
    ax.plot(range(K), profile, marker=marker, markersize=4, label=name, linewidth=1.2)

ax.set_xlabel('q-direction index k')
ax.set_ylabel('Mean voxel intensity')
ax.set_title('Per-direction mean intensity — all methods')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5.5 FBP Round-Trip: NN → SH → NN

Tests how much angular information survives projection through a band-limited spherical
harmonic basis.  The FBP result (already in NN format, one coefficient per direction per
voxel) is lifted to SH coefficients via Driscoll–Healy quadrature, then evaluated back
at the K Fibonacci directions to produce a round-tripped `(K, X, Y, Z)` volume.

The round trip is repeated for each `ell_max` in `RT_ELL_MAX_VALUES`.  Lower orders act
as an angular low-pass filter and reveal the smoothing imposed by the SH basis — the
same regularisation that the mumott SH reconstruction implicitly applies.

The interactive viewer lets you:
- **Dropdown** — switch between the original FBP and each round-trip variant
- **Diff from FBP** — toggle a difference map (round-trip − FBP) to isolate what the
  SH projection adds or removes
- **k slider** — step through the K q-directions independently
- **x / y / z sliders** — navigate the three orthogonal planes; crosshairs
  (green = x, red = y, blue = z) mark the position of the other two slices

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
RT_ELL_MAX_VALUES = [6, 8, 10]
# ─────────────────────────────────────────────────────────────────────────────

# recon_fbp_np is (K, X, Y, Z) after spherical crop; NN→SH expects (X, Y, Z, K)
fbp_xyzk = recon_fbp_np.transpose(1, 2, 3, 0)

# NearestNeighbor basis for NN→SH step (basis_nn may not exist if loaded from cache)
_basis_nn_rt = NearestNeighbor(
    directions=y_directions,
    probed_coordinates=geometry.probed_coordinates,
)

# ProbedCoordinates for back-evaluation at the K Fibonacci directions
pc_rt = ProbedCoordinates()
pc_rt.vector = y_directions[:, np.newaxis, np.newaxis, :]  # (K, 1, 1, 3)

recon_rt = {}
for _ell in RT_ELL_MAX_VALUES:
    # Step 1: NN → SH  (Driscoll-Healy quadrature, resolution independent of K)
    _coeffs_rt = _basis_nn_rt.get_spherical_harmonic_coefficients(fbp_xyzk, ell_max=_ell)
    # Step 2: SH → NN  (evaluate SH basis at K Fibonacci directions)
    _basis_sh_rt = SphericalHarmonics(ell_max=_ell, probed_coordinates=geometry.probed_coordinates)
    _eval_rt = _basis_sh_rt._get_projection_matrix(pc_rt)[:, 0, 0, :]  # (K, C)
    recon_rt[_ell] = np.einsum('xyzc, kc -> kxyz', _coeffs_rt, _eval_rt)  # (K, X, Y, Z)
    print(f'ell_max={_ell:2d}: SH coeffs {_coeffs_rt.shape} → round-trip {recon_rt[_ell].shape}')


In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
from matplotlib.lines import Line2D

_crosshair_handles = [Line2D([], [], color=c, lw=1.5)
                      for c in ['limegreen', 'tomato', 'deepskyblue']]
_crosshair_labels  = ['x', 'y', 'z']


def _make_rt_viewer(rt_volumes, ref_volume, sphere_mask, directions):
    K_, X_, Y_, Z_ = ref_volume.shape

    # Color scale anchored to FBP sphere interior (fixed across all dropdown options)
    mean_ref = ref_volume.mean(axis=0)
    _abs_lo  = float(np.percentile(mean_ref[sphere_mask], 2))
    _abs_hi  = float(np.percentile(mean_ref[sphere_mask], 98))
    _diff_lim = 0.15 * (_abs_hi - _abs_lo)

    options = {'FBP (original)': ref_volume}
    options.update({f'RT ℓ_max={l}': v for l, v in rt_volumes.items()})
    keys = list(options.keys())

    # Pre-extract sphere interiors once for speed.
    interiors = [ref_volume[k_][sphere_mask].ravel() for k_ in range(K_)]
    all_vals  = np.concatenate(interiors)
    bins = np.linspace(np.percentile(all_vals, 0.5), np.percentile(all_vals, 99.5), 80)

    def _view(dataset, show_difference, k, x, y, z):
        vol   = options[dataset]
        vol_k = vol[k]

        if show_difference and dataset != 'FBP (original)':
            data   = vol_k - ref_volume[k]
            cmap   = 'RdBu_r'
            lo, hi = -_diff_lim, _diff_lim
            clabel = 'Round-trip − FBP'
        else:
            data   = vol_k
            cmap   = 'inferno'
            lo, hi = _abs_lo, _abs_hi
            clabel = 'Intensity'

        fig = plt.figure(figsize=(13, 9))
        gs  = fig.add_gridspec(2, 3, height_ratios=[1, 0.65], hspace=0.45, wspace=0.3)
        ax_yz   = fig.add_subplot(gs[0, 0])
        ax_xz   = fig.add_subplot(gs[0, 1])
        ax_xy   = fig.add_subplot(gs[0, 2])
        ax_hist = fig.add_subplot(gs[1, :])

        kw = dict(cmap=cmap, vmin=lo, vmax=hi, aspect='equal', origin='lower')

        # YZ plane (x fixed) — Y horizontal, Z vertical
        im = ax_yz.imshow(data[x, :, :].T, **kw)
        ax_yz.axvline(y, color='tomato',      lw=0.9, alpha=0.85)
        ax_yz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_yz.set_title(f'YZ  (x={x})')
        ax_yz.set_xlabel('Y')
        ax_yz.set_ylabel('Z')

        # XZ plane (y fixed) — X horizontal, Z vertical
        ax_xz.imshow(data[:, y, :].T, **kw)
        ax_xz.axvline(x, color='limegreen',   lw=0.9, alpha=0.85)
        ax_xz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_xz.set_title(f'XZ  (y={y})')
        ax_xz.set_xlabel('X')
        ax_xz.set_ylabel('Z')

        # XY plane (z fixed) — X horizontal, Y vertical
        ax_xy.imshow(data[:, :, z].T, **kw)
        ax_xy.axvline(x, color='limegreen', lw=0.9, alpha=0.85)
        ax_xy.axhline(y, color='tomato',    lw=0.9, alpha=0.85)
        ax_xy.set_title(f'XY  (z={z})')
        ax_xy.set_xlabel('X')
        ax_xy.set_ylabel('Y')

        ax_xy.legend(_crosshair_handles, _crosshair_labels,
                     fontsize=7, loc='lower right', framealpha=0.5, title='slice pos.')
        plt.colorbar(im, ax=[ax_yz, ax_xz, ax_xy], shrink=0.65, label=clabel)

        # Histogram: sphere-interior distributions for all K overlaid.
        # Always uses raw vol (not diff) so we see actual intensities regardless
        # of the show_difference toggle — normalization diagnostic is about the
        # raw scale, not residuals.
        vol_interiors = [vol[k_][sphere_mask].ravel() for k_ in range(K_)]

        for k_ in range(K_):
            ax_hist.hist(
                vol_interiors[k_], bins=bins, histtype='step',
                color='steelblue' if k_ == k else 'gray',
                lw=1.8        if k_ == k else 0.5,
                alpha=1.0     if k_ == k else 0.25,
                label=f'k={k_} (selected)' if k_ == k else '_nolegend_',
            )

        mu    = float(np.mean(vol_interiors[k]))
        sigma = float(np.std(vol_interiors[k]))
        ax_hist.axvline(mu,         color='steelblue', lw=1.5, ls='--',
                        label=f'μ = {mu:.3g}')
        ax_hist.axvline(mu - sigma, color='steelblue', lw=1.0, ls=':',
                        label=f'σ = {sigma:.3g}')
        ax_hist.axvline(mu + sigma, color='steelblue', lw=1.0, ls=':')
        # Colourscale limits — if distributions are far outside these, the
        # colourmap is clipping and normalization may be mismatched across K.
        ax_hist.axvline(_abs_lo, color='orange', lw=1.2, ls='--', alpha=0.8,
                        label='cmap lo / hi')
        ax_hist.axvline(_abs_hi, color='orange', lw=1.2, ls='--', alpha=0.8)

        ax_hist.set_xlabel('Voxel intensity (sphere interior)')
        ax_hist.set_ylabel('Count')
        ax_hist.set_title(
            f'Sphere-interior distributions — all K overlaid  '
            f'(highlighted: k={k},  μ={mu:.3g},  σ={sigma:.3g})'
        )
        ax_hist.legend(fontsize=8)

        plt.suptitle(
            f'{dataset}  |  k={k}  y_dir={np.round(directions[k], 2)}',
            fontsize=11,
        )
        plt.show()

    interact(
        _view,
        dataset=widgets.Dropdown(
            options=keys, value=keys[0], description='Volume:',
            style={'description_width': 'initial'},
        ),
        show_difference=widgets.Checkbox(value=False, description='Diff from FBP'),
        k=widgets.IntSlider(min=0, max=K_-1, step=1, value=0,
                            description='k (dir)', continuous_update=False),
        x=widgets.IntSlider(min=0, max=X_-1, step=1, value=X_//2,
                            description='x', continuous_update=False),
        y=widgets.IntSlider(min=0, max=Y_-1, step=1, value=Y_//2,
                            description='y', continuous_update=False),
        z=widgets.IntSlider(min=0, max=Z_-1, step=1, value=Z_//2,
                            description='z', continuous_update=False),
    )


_make_rt_viewer(recon_rt, recon_fbp_np, sphere_mask_np, y_directions)


## 6. IsoNet Pipeline — Missing-Wedge Correction

Loads the output of `run_pipeline` (`final/` directory) and compares it to the raw FBP.

Set `ISONET_FINAL_DIR` to the `final/` subdirectory of the `output_dir` you passed to
`run_pipeline`.  The section is skipped gracefully if the path does not exist yet.

- **6.1** Side-by-side mean-intensity slices: FBP vs IsoNet
- **6.2** Difference map (IsoNet − FBP): what the model filled in
- **6.3** Fourier-space comparison: the dark missing-wedge stripes should shrink
- **6.4** Per-direction profile: directions with large missing wedge should gain intensity

In [ ]:
import pathlib, sys
sys.path.insert(0, '/myhome/smartt')

# ── Point this to your pipeline output_dir / 'final' ─────────────────────────
# ISONET_FINAL_DIR = pathlib.Path('/myhome/data/smartt/shared/results/saxs_isonet/final')
ISONET_FINAL_DIR = pathlib.Path('/myhome/data/smartt/shared/isonet_results/frogbone_fbp/round_1_volumes')
ALPHA_DEG        = 45.0    # must match what was passed to run_pipeline
# ─────────────────────────────────────────────────────────────────────────────

if not ISONET_FINAL_DIR.exists():
    print(f'[SKIP] {ISONET_FINAL_DIR} does not exist yet.')
    print('Run run_pipeline(...) first and set ISONET_FINAL_DIR above.')
    ISONET_AVAILABLE = False
else:
    vol_paths = sorted(ISONET_FINAL_DIR.glob('vol_*.npy'))
    if len(vol_paths) != K:
        print(f'[WARN] Found {len(vol_paths)} volumes, expected K={K}. Check ISONET_FINAL_DIR.')
        ISONET_AVAILABLE = False
    else:
        recon_isonet = np.stack([np.load(p) for p in vol_paths], axis=0)  # (K, X, Y, Z)
        # Apply sphere masking to match the cropped base reconstructions.
        from smartt.saxs_isonet.preprocess import spherical_crop, make_sphere_mask
        _cube = SPHERE_D if 'SPHERE_D' in dir() else None
        recon_isonet, _, _iso_d = spherical_crop(
            torch.from_numpy(recon_isonet.astype(np.float32)), cube_size=_cube
        )
        recon_isonet = recon_isonet.numpy()
        if 'sphere_mask_np' not in dir():
            sphere_mask_np = make_sphere_mask(_iso_d)
        print(f'Loaded IsoNet output: shape={recon_isonet.shape}  '
              f'range=[{recon_isonet.min():.3e}, {recon_isonet.max():.3e}]')
        ISONET_AVAILABLE = True

from smartt.saxs_isonet.wedge import all_missing_arcs, goniometer_axis_for_half_space
G_AXIS = goniometer_axis_for_half_space(HALF_SPACE)
arcs_deg = np.degrees(all_missing_arcs(y_directions, ALPHA_DEG, G_AXIS))

In [ ]:
reconstructions = {
    'FBP (smartt)':               recon_fbp_np,
    'ISONET (smartt)':      recon_isonet,
    'Gaussian Kernel (mumott)':   recon_gk,
}

vol_shape = recon_fbp_np.shape[1:]
mid_x, mid_y, mid_z = [s // 2 for s in vol_shape]
print('Volume shape:', vol_shape)
print(f'Central slices: x={mid_x}, y={mid_y}, z={mid_z}')


In [ ]:
K_INSPECT = 0   # change to inspect a different direction

r_fbp_k = recon_fbp_np[K_INSPECT]
vmin_k  = r_fbp_k.min()
vmax_k  = np.percentile(recon_isonet, 99.8)

fig, axes = plt.subplots(len(reconstructions), 3, figsize=(15, 4 * n_methods))
for row, (name, recon) in enumerate(reconstructions.items()):
    vol    = recon[K_INSPECT]
    slices = [vol[mid_x, :, :], vol[:, mid_y, :], vol[:, :, mid_z]]
    titles = [f'{name} YZ  x={mid_x}', f'{name} XZ  y={mid_y}', f'{name}  XY  z={mid_z}']
    for col, (sl, title) in enumerate(zip(slices, titles)):
        im = axes[row, col].imshow(sl, cmap='inferno', vmin=vmin_k, vmax=vmax_k)
        axes[row, col].axis('off')
        axes[row, col].set_title(title, fontsize=10)
    axes[row, 0].set_ylabel(name, fontsize=10)
    

# plt.colorbar(im, ax=axes[:, 2], shrink=0.5, label='Intensity')
plt.suptitle(
    f'Direction k={K_INSPECT}  y={np.round(y_directions[K_INSPECT], 3)}',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
if not ISONET_AVAILABLE:
    print('[SKIP] IsoNet output not available.')
else:
    # ── 6.1  Side-by-side mean-intensity slices: FBP vs IsoNet ───────────────
    pair = {'FBP (smartt)': recon_fbp_np, 'IsoNet (smartt)': recon_isonet}

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for row, (name, recon) in enumerate(pair.items()):
        print("name", name)
        mean_vol = recon.mean(axis=0)
        for col, (sl, title) in enumerate(zip(
            [mean_vol[mid_x, :, :], mean_vol[:, mid_y, :], mean_vol[:, :, mid_z]],
            [f'{name} YZ  x={mid_x}', f'{name} XZ  y={mid_y}', f'{name} XY  z={mid_z}'],
        )):
            im = axes[row, col].imshow(sl, cmap='inferno', vmin=vmin, vmax=vmax)
            axes[row, col].axis('off')
            axes[row, col].set_title(title, fontsize=10)
        axes[row, 0].set_ylabel(name, fontsize=10)
    # plt.colorbar(im, ax=axes[:, 2], shrink=0.5, label='Mean intensity')
    plt.suptitle('6.1  Mean scattering intensity — FBP vs IsoNet (same colour scale)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
if not ISONET_AVAILABLE:
    print('[SKIP] IsoNet output not available.')
else:
    # ── 6.2  Difference map: IsoNet − FBP ────────────────────────────────────
    # Positive (red)  = IsoNet adds intensity the FBP was missing.
    # Negative (blue) = IsoNet suppresses FBP artefacts.
    diff_mean = recon_isonet.mean(axis=0) - mean_fbp
    absmax    = np.percentile(np.abs(diff_mean), 98)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, sl, title in zip(axes,
        [diff_mean[mid_x, :, :], diff_mean[:, mid_y, :], diff_mean[:, :, mid_z]],
        [f'{title} YZ  x={mid_x}', f'{title} XZ  y={mid_y}', f'{title} XY  z={mid_z}'],
    ):
        im = ax.imshow(sl, cmap='RdBu_r', vmin=-absmax, vmax=absmax)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.colorbar(im, ax=axes, shrink=0.7, label='IsoNet − FBP')
    plt.suptitle('6.2  Difference map (IsoNet − FBP mean intensity)\n'
                 'Red = recovered signal | Blue = suppressed artefact',
                 fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
if not ISONET_AVAILABLE:
    print('[SKIP] IsoNet output not available.')
else:
    # ── 6.3  Fourier-space comparison — the key missing-wedge diagnostic ──────
    # For each inspected direction the dark wedge in log|FFT| should be narrower
    # in the IsoNet output than in the raw FBP.
    # Show directions with different missing-wedge sizes for context.
    sort_idx  = np.argsort(arcs_deg)           # ascending: smallest wedge first
    k_small   = int(sort_idx[0])               # near-pole: smallest missing wedge
    k_medium  = int(sort_idx[len(sort_idx)//2])
    k_large   = int(sort_idx[-1])              # near-equator: largest missing wedge
    k_show    = [k_small, k_medium, k_large]
    k_labels  = ['small wedge\n(near pole)', 'medium wedge', 'large wedge\n(near equator)']

    fig, axes = plt.subplots(len(k_show) * 2, 2, figsize=(10, 5 * len(k_show)))

    for row_base, (k, klabel) in enumerate(zip(k_show, k_labels)):
        for col, (name, recon) in enumerate([('FBP', recon_fbp_np), ('IsoNet', recon_isonet)]):
            vol_k   = recon[k].astype(np.float32)
            fft_vol = np.abs(np.fft.fftshift(np.fft.fftn(vol_k)))
            log_fft = np.log1p(fft_vol)
            fx, fy, fz = [s // 2 for s in fft_vol.shape]

            # Two Fourier slices: XZ and YZ
            for sub_row, (sl, plane) in enumerate([(log_fft[:, fy, :], 'XZ'), (log_fft[fx, :, :], 'YZ')]):
                ax = axes[row_base * 2 + sub_row, col]
                vlo, vhi = np.percentile(sl, [5, 99])
                ax.imshow(sl, cmap='hot', aspect='auto', vmin=vlo, vmax=vhi)
                ax.axis('off')
                if sub_row == 0:
                    ax.set_title(f'{name}', fontsize=9)
                if col == 0:
                    ax.set_ylabel(
                        f'k={k}  {klabel}\nmissing={arcs_deg[k]:.0f}°\n({plane})',
                        fontsize=7,
                    )

    plt.suptitle('6.3  log|FFT| comparison — missing wedge (dark region) should shrink in IsoNet',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
if not ISONET_AVAILABLE:
    print('[SKIP] IsoNet output not available.')
else:
    # ── 6.4  Per-direction mean intensity: FBP vs IsoNet vs missing arc ───────
    # If the pipeline recovers missing-wedge content, directions with a large
    # missing arc (equatorial) should gain more intensity than near-pole directions.
    fbp_profile    = recon_fbp_np.mean(axis=(1, 2, 3))    # (K,)
    isonet_profile = recon_isonet.mean(axis=(1, 2, 3))    # (K,)
    gain           = isonet_profile - fbp_profile          # (K,)

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

    # Top: absolute profiles
    axes[0].plot(range(K), fbp_profile,    'o-', ms=4, lw=1.2, label='FBP')
    axes[0].plot(range(K), isonet_profile, 's-', ms=4, lw=1.2, label='IsoNet')
    axes[0].set_ylabel('Mean voxel intensity')
    axes[0].set_title('6.4  Per-direction mean intensity — FBP vs IsoNet')
    axes[0].legend(fontsize=9)

    # Bottom: gain coloured by missing arc size
    sc = axes[1].scatter(range(K), gain, c=arcs_deg, cmap='RdYlGn_r', s=50, zorder=3)
    axes[1].axhline(0, color='k', lw=0.8, ls='--')
    axes[1].set_xlabel('q-direction index k  (sorted by Fibonacci grid)')
    axes[1].set_ylabel('Gain (IsoNet − FBP)')
    axes[1].set_title('Intensity gain per direction (coloured by missing arc size)\n'
                      'Equatorial directions (red) should show larger positive gain')
    plt.colorbar(sc, ax=axes[1], label='Missing arc (degrees)')

    plt.tight_layout()
    plt.show()

    # Summary statistics
    print(f'Mean gain  : {gain.mean():+.3e}')
    print(f'Max gain   : {gain.max():+.3e}  (k={gain.argmax()}, arc={arcs_deg[gain.argmax()]:.1f}°)')
    corr = np.corrcoef(arcs_deg, gain)[0, 1]
    print(f'Correlation(missing_arc, gain): {corr:.3f}  '
          f'(positive → equatorial dirs gain more, as expected)')